# Unidade 2 - Bloco prático da Aula 01: SMOTE dentro e fora do pipeline

Reproduz, em escala de laboratório, o erro dos estudos de parto prematuro: o mesmo SMOTE aplicado antes da validação cruzada (errado) e dentro do pipeline (certo), com um conjunto de teste intocado como juiz. Compare os quatro números impressos: a promessa de cada arranjo e o desempenho real de cada um.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

X, y = make_classification(n_samples=2000, n_features=15, n_informative=6,
                           weights=[0.9, 0.1], flip_y=0.02, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ERRADO: SMOTE aplicado ao treino inteiro ANTES da validacao cruzada
X_res, y_res = SMOTE(random_state=42).fit_resample(X_tr, y_tr)
f1_cv_errado = cross_val_score(LogisticRegression(max_iter=1000),
                               X_res, y_res, cv=cv, scoring="f1").mean()

# CERTO: SMOTE dentro do pipeline, refeito no treino de cada dobra
pipe = ImbPipeline([("smote", SMOTE(random_state=42)),
                    ("clf", LogisticRegression(max_iter=1000))])
f1_cv_certo = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring="f1").mean()

# A verdade: desempenho no teste intocado
m_errado = LogisticRegression(max_iter=1000).fit(X_res, y_res)
f1_te_errado = f1_score(y_te, m_errado.predict(X_te))
pipe.fit(X_tr, y_tr)
f1_te_certo = f1_score(y_te, pipe.predict(X_te))

print(f"CV com SMOTE antes (errado): F1 = {f1_cv_errado:.3f}")
print(f"Teste real desse modelo:     F1 = {f1_te_errado:.3f}")
print(f"CV com SMOTE no pipeline:    F1 = {f1_cv_certo:.3f}")
print(f"Teste real desse modelo:     F1 = {f1_te_certo:.3f}")